# Web Crawler Demo: CNN

This notebook demonstrates the WebCrawler Spider by crawling CNN.com and analyzing the results.

In [5]:
import logging

from WebCrawler import Spider

# Configure logging to see crawler activity
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

## Set up and run the Spider

We'll crawl CNN with a depth of 1 (homepage + links from the homepage) to keep it manageable.

In [6]:
# Create Spider instance
start_url = "https://www.cnn.com"
max_depth = 1  # Keep it shallow to avoid excessive requests

spider = Spider(start_url=start_url, max_depth=max_depth, debug=True)

print(f"Starting crawl of {start_url} (max depth: {max_depth})...\n")

# Run the async crawler (await works in Jupyter)
documents = await spider.run_async()

print(f"\nCrawl complete! Visited {len(documents)} pages.")

2026-06-07 22:02:13,815 - WebCrawler.Crawler - DEBUG - Fetching https://www.cnn.com
2026-06-07 22:02:13,900 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-07 22:02:13,900 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-07 22:02:13,900 WebCrawler.Spider INFO     Visited: https://www.cnn.com (Total Visited: 1)
2026-06-07 22:02:13,900 - WebCrawler.Spider - INFO - Visited: https://www.cnn.com (Total Visited: 1)


Starting crawl of https://www.cnn.com (max depth: 1)...


Crawl complete! Visited 1 pages.


## Analyze Results

In [7]:
# Display page titles and link counts
print(f"{'URL':<60} {'Title':<40} {'Links':<8}")
print("-" * 110)

for doc in documents:
    url_short = doc.url[:59] if len(doc.url) > 59 else doc.url
    title_short = doc.title[:39] if len(doc.title) > 39 else doc.title
    num_links = len(doc.links)
    print(f"{url_short:<60} {title_short:<40} {num_links:<8}")

URL                                                          Title                                    Links   
--------------------------------------------------------------------------------------------------------------
https://www.cnn.com                                          Breaking News, Latest News and Videos |  232     


## Statistics

In [8]:
# Aggregate statistics
total_links = sum(len(doc.links) for doc in documents)
total_internal = sum(len(doc.internal_links) for doc in documents)
total_external = sum(len(doc.external_links) for doc in documents)

print(f"Total pages crawled: {len(documents)}")
print(f"Total links found: {total_links}")
print(f"  - Internal: {total_internal}")
print(f"  - External: {total_external}")
print(f"\nAverage links per page: {total_links / len(documents):.1f}")

Total pages crawled: 1
Total links found: 232
  - Internal: 207
  - External: 25

Average links per page: 232.0


## Sample External Links from Homepage

In [9]:
# Show external links from the first page (homepage)
if documents:
    homepage = documents[0]
    print(f"External links from {homepage.url}:\n")
    for link in homepage.external_links[:10]:  # Show first 10
        print(f"  • {link.url}")
    if len(homepage.external_links) > 10:
        print(f"  ... and {len(homepage.external_links) - 10} more")

External links from https://www.cnn.com:

  • https://us.cnn.com?hpt=header_edition-picker
  • https://edition.cnn.com?hpt=header_edition-picker
  • https://arabic.cnn.com?hpt=header_edition-picker
  • https://cnnespanol.cnn.com/?hpt=header_edition-picker
  • https://bleacherreport.com/
  • https://www.cnn10.com
  • https://cnn.it/5thingsquiz
  • https://careers.wbd.com/cnnjobs
  • https://facebook.com/CNN
  • https://twitter.com/CNN
  ... and 15 more


## Sample Internal Links from Homepage

In [10]:
# Show internal links from the first page (homepage)
if documents:
    homepage = documents[0]
    print(f"Internal links from {homepage.url}:\n")
    for link in homepage.internal_links[:10]:  # Show first 10
        text_preview = (link.text[:40] + "...") if len(link.text) > 40 else link.text
        print(f"  • {link.url}")
        if text_preview:
            print(f"    → {text_preview}")
    if len(homepage.internal_links) > 10:
        print(f"  ... and {len(homepage.internal_links) - 10} more")

Internal links from https://www.cnn.com:

  • https://www.cnn.com/us
    → US
  • https://www.cnn.com/world
    → World
  • https://www.cnn.com/politics
    → Politics
  • https://www.cnn.com/business
    → Business
  • https://www.cnn.com/health
    → Health
  • https://www.cnn.com/entertainment
    → Entertainment
  • https://www.cnn.com/cnn-underscored
    → Underscored
  • https://www.cnn.com/style
    → Style
  • https://www.cnn.com/travel
    → Travel
  • https://www.cnn.com/sports
    → Sports
  ... and 197 more


## Domain Analysis

In [ ]:
# Extract domain information using tldextract
from collections import Counter

# Get domains from external links
external_domains = Counter()
for doc in documents:
    for link in doc.external_links:
        try:
            external_domains[link.url.split("/")[2]] += 1  # Extract domain from URL
        except Exception:
            pass

print("Most common external domains:")
for domain, count in external_domains.most_common(10):
    print(f"  {domain}: {count} links")